# Text Expert — SCITSR Dataset

This is a file that produces the text expert cell predictions.

This file is specifically designed for working through the SciTSR dataset to go through the images in that folder

In [ ]:
from pathlib import Path
import json, re, time
import base64

from tqdm import tqdm

import csv
from openai import OpenAI
from pydantic import BaseModel
from typing import List

In [ ]:

DATA_ROOT = Path("input")
SPLITS = ["test"]         

OUT_DIR = Path("data/outputs/text_expert_gemma4_SciTSR_test_run")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "google/gemma-4-31b-it" 

# you should change this to what ever llm you would like to use
client_2 = OpenAI(
    api_key="vllm",                    
    base_url="http://localhost:8001/v1"
)


SLEEP_BETWEEN_CALLS_SEC = 0.35
MAX_RETRIES             = 1
RETRY_BACKOFF_SEC       = 2.0

print("DATA_ROOT:", DATA_ROOT.resolve())
print("OUT_DIR  :", OUT_DIR.resolve())


CELL_SCHEMA = {
    "type": "object",
    "properties": {
        "cells": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "sr": {"type": "integer"},
                    "er": {"type": "integer"},
                    "sc": {"type": "integer"},
                    "ec": {"type": "integer"},
                    "text": {"type": "string"},
                },
                "required": ["sr", "er", "sc", "ec", "text"],
            },
        }
    },
    "required": ["cells"],
}

class Cell(BaseModel):
    sr: int
    er: int
    sc: int
    ec: int
    text: str

class CellSchema(BaseModel):
    cells: List[Cell]

In [ ]:
prompt = """
    You are an expert LaTeX table parser. Analyze this LaTeX table markup and convert it to a structured format.

        ## LEARNING EXAMPLES - Common Mistakes to Avoid:

        ### Example 1: Incorrect Multicolumn Span Calculation
        BAD LaTeX INPUT: 
        \\begin{{tabular}}{{|c|c|c|c|}}
        \\hline
        \\multicolumn{{3}}{{|c|}}{{Material Properties}} & Test \\\\
        \\hline
        Steel & 200 & GPa & Pass \\\\
        \\end{{tabular}}
        
        WRONG OUTPUT:
        [{{"text": "Material Properties", "start_row": 0, "end_row": 0, "start_col": 0, "end_col": 0}}]
        
        WHAT'S WRONG: Failed to calculate multicolumn span - \\multicolumn{{3}} means span 3 columns
        
        CORRECT OUTPUT:
        [{{"text": "Material Properties", "start_row": 0, "end_row": 0, "start_col": 0, "end_col": 2}},
         {{"text": "Test", "start_row": 0, "end_row": 0, "start_col": 3, "end_col": 3}}]

        ### Example 2: Mathematical Symbols and Greek Letters Not Converted
        BAD LaTeX INPUT:
        \\begin{{tabular}}{{cc}}
        Compound & $K_{{eq}} = 1.5 \\times 10^{{-3}}$ \\\\
        $\\alpha$-Fe$_2$O$_3$ & $\\Delta G = -25.3 \\pm 0.5$ kJ/mol \\\\
        \\end{{tabular}}
        
        WRONG OUTPUT:
        [{{"text": "$K_{{eq}} = 1.5 \\times 10^{{-3}}$", "start_row": 0, "end_row": 0, "start_col": 1, "end_col": 1}},
         {{"text": "$\\alpha$-Fe$_2$O$_3$", "start_row": 1, "end_row": 1, "start_col": 0, "end_col": 0}}]
        
        WHAT'S WRONG: LaTeX math symbols not converted to Unicode
        
        CORRECT OUTPUT:
        [{{"text": "Keq = 1.5 × 10^⁻³", "start_row": 0, "end_row": 0, "start_col": 1, "end_col": 1}},
         {{"text": "α-Fe_{{2}}O_{{3}}", "start_row": 1, "end_row": 1, "start_col": 0, "end_col": 0}},
         {{"text": "ΔG = -25.3 ± 0.5 kJ/mol", "start_row": 1, "end_row": 1, "start_col": 1, "end_col": 1}}]

        ### Example 3: Combined Multirow/Multicolumn + Math Symbol Errors  
        BAD LaTeX INPUT:
        \\begin{{tabular}}{{|c|c|c|}}
        \\hline
        \\multirow{{2}}{{*}}{{$\\beta$-phase}} & \\multicolumn{{2}}{{c|}}{{Properties}} \\\\
        & $\\sigma_y$ (MPa) & $E$ (GPa) \\\\
        \\hline
        Ti-6Al-4V & $924 \\pm 15$ & $114 \\pm 2$ \\\\
        \\end{{tabular}}
        
        WRONG OUTPUT:
        [{{"text": "$\\beta$-phase", "start_row": 0, "end_row": 0, "start_col": 0, "end_col": 0}},
         {{"text": "Properties", "start_row": 0, "end_row": 0, "start_col": 1, "end_col": 1}}]
        
        WHAT'S WRONG: 1) \\multirow{{2}} span ignored, 2) \\multicolumn{{2}} span wrong, 3) Math symbols not converted
        
        CORRECT OUTPUT:
        [{{"text": "β-phase", "start_row": 0, "end_row": 1, "start_col": 0, "end_col": 0}},
         {{"text": "Properties", "start_row": 0, "end_row": 0, "start_col": 1, "end_col": 2}},
         {{"text": "σy (MPa)", "start_row": 1, "end_row": 1, "start_col": 1, "end_col": 1}},
         {{"text": "E (GPa)", "start_row": 1, "end_row": 1, "start_col": 2, "end_col": 2}},
         {{"text": "Ti-6Al-4V", "start_row": 2, "end_row": 2, "start_col": 0, "end_col": 0}},
         {{"text": "924 ± 15", "start_row": 2, "end_row": 2, "start_col": 1, "end_col": 1}},
         {{"text": "114 ± 2", "start_row": 2, "end_row": 2, "start_col": 2, "end_col": 2}}]

        ## NOW ANALYZE THIS ACTUAL LaTeX TABLE:
        {latex_content}

        ## CRITICAL SPAN HANDLING:
        - \\multicolumn{{3}}{{...}}{{text}} means cell spans 3 columns: if at col 1, then start_col=1, end_col=3
        - \\multirow{{2}}{{...}}{{text}} means cell spans 2 rows: if at row 0, then start_row=0, end_row=1
        - Combined spans: \\multicolumn{{2}}{{...}}{{\\multirow{{3}}{{...}}{{text}}}} spans 2 cols and 3 rows

        ## SPAN EXAMPLES:
        - Normal cell: start_row=0, end_row=0, start_col=0, end_col=0
        - Spans 3 columns: start_row=0, end_row=0, start_col=1, end_col=3
        - Spans 2 rows: start_row=1, end_row=2, start_col=0, end_col=0
        - Spans 2 cols + 3 rows: start_row=0, end_row=2, start_col=1, end_col=2

        ## Your task:
        1. Parse the table structure (rows, columns, spans)
        2. Extract the actual content as it would appear in the final document
        3. Convert LaTeX commands to their natural representation:
           - \\textbf{{text}} → text (keep the text, ignore bold formatting)
           - \\textit{{text}} → text (keep the text, ignore italic formatting)
           - $math$ → math (convert to natural symbols)
           - \\pm → ±
           - \\times → ×
           - \\alpha → α, \\beta → β, etc.
           - \\deg → °
           - Percentages, temperatures, uncertainties as natural text
        4. **MOST IMPORTANT**: Properly calculate start_row, end_row, start_col, end_col for spanned cells
        5. For spanned cells, DO NOT create separate entries for covered positions
        
        ## CRITICAL: 
        - If a cell spans multiple rows/columns, the end_row/end_col MUST be different from start_row/start_col
        - Calculate spans based on \\multicolumn{{n}} and \\multirow{{n}} commands
        - Do not create duplicate cells for positions covered by spans
        - Output natural, readable content (not LaTeX commands)
        - Preserve mathematical symbols and scientific notation

        ## OUTPUT FORMAT:
        Respond ONLY in VALID JSON format. DO NOT include any explanations or extra text.
            {"cells": [
            {"sr": 0, "er": 0, "sc": 0, "ec": 0, "text": "cell content"},
            {"sr": 0, "er": 0, "sc": 1, "ec": 1, "text": "cell content"},
            {"sr": 0, "er": 2, "sc": 1, "ec": 1, "text": "cell content"}
            .....
            ]}

        ONLY RETURN THE OUTPUT. NO OTHER CONTENT SHOULD BE RETURNED

    """

In [ ]:
def encode_image(image_path: Path) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def normalize_text(text: str) -> str:
    
    if text is None:
        return ""

    text = str(text).lower()
    text = text.replace("\\times", "x")
    text = text.replace(chr(0x2212), "-").replace(chr(0x2013), "-").replace(chr(0x2014), "-")
    text = re.sub(r"\$\^\{(\d+)\}\$", r"\1", text)
    text = text.replace("$", "")

    return re.sub(r"\s+", " ", text).strip()


def parse_gt_json(json_path: Path, normalize: bool = True) -> dict:
    data = json.loads(json_path.read_text(encoding="utf-8"))
    cells = data.get("cells", [])

    gt_cells = {}
    for cell in cells:

        sr = cell.get("start_row")
        sc = cell.get("start_col")

        if sr is None or sc is None:
            continue

        
        text = cell.get("text") or ""
        if not text and cell.get("content"):
            text = cell["content"][0] if cell["content"] else ""

        text = normalize_text(text) if normalize else (text or "")
        gt_cells[(int(sr), int(sc))] = text

    return gt_cells


def parse_pred_json(pred_json) -> dict:
    
    pred_cells = {}
    pred_json = pred_json.get("cells", []) if isinstance(pred_json, dict) else pred_json

    for cell in pred_json:

        sr = cell.get("sr", cell.get("start-row"))
        sc = cell.get("sc", cell.get("start-col"))

        if sr is None or sc is None:
            continue

        text = normalize_text(cell.get("text", ""))
        pred_cells[(int(sr), int(sc))] = text

    return pred_cells

In [ ]:
def text_agent(prompt_text: str, nougat: str) -> CellSchema:
    user_prompt = f"""
        Now, please extract the table cells from this image and return in the specified format: 

        {nougat}
    """

    response = client_2.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": prompt_text},
            {
                "role": "user",
                "content": [
                    {"type": "text",
                     "text": user_prompt},
                ],
            },
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "cell-info",
                "schema": CELL_SCHEMA
            }
        },
        temperature=0.0,
        top_p=0.95,
        max_tokens=16384,
        extra_body={
            "chat_template_kwargs": {"enable_thinking": False}
        }
    )
    return response.choices[0].message.content


In [ ]:
def discover_splits(data_root: Path) -> list:
    splits = []
    for p in sorted(data_root.iterdir()):
        if p.is_dir() and (p / "img").exists() and (p / "structure_processed").exists():
            splits.append(p.name)

    return splits


if not SPLITS:
    SPLITS = discover_splits(DATA_ROOT)

print("SPLITS:", SPLITS)


def list_pairs_for_split(split_name: str) -> list:
    split_dir = DATA_ROOT / split_name
    img_dir = split_dir / "img"
    gt_dir = split_dir / "structure_processed"

    pairs = []
    for img_path in sorted(img_dir.glob("*.png")):
        gt_path = gt_dir / f"{img_path.stem}.json"
        pairs.append((img_path, gt_path if gt_path.exists() else None))

    print(f"[{split_name}]  images found: {len(pairs)}, "
          f"with GT: {sum(1 for _, g in pairs if g is not None)}")
    return pairs

In [ ]:
def process_pair_sequential(img_path, gt_json_path, split_name, preds_dir, errs_dir):
    stem = img_path.stem
    pred_file = preds_dir / f"{stem}.json"
    raw_output = None

    if pred_file.exists():
        try:

            pred_json = json.loads(pred_file.read_text(encoding="utf-8"))
            gt_cells = parse_gt_json(gt_json_path, normalize=True)
            pred_cells = parse_pred_json(pred_json)

            return {
                "split": split_name, 
                "stem": stem,
                "image_path": str(img_path), 
                "gt_path": str(gt_json_path),
                "num_gt_cells": len(gt_cells), 
                "num_pred_cells": len(pred_cells),
                "status": "ok",
            }
        except Exception:
            pass 

    if gt_json_path is None:
        return {
            "split": split_name, 
            "stem": stem,
            "image_path": str(img_path), 
            "gt_path": "",
            "num_gt_cells": 0, 
            "num_pred_cells": 0,
            "status": "missing_gt",
        }

    
    tex_path = Path(f"./input/test/nougat/{stem}.txt")
    nougat = tex_path.read_text(encoding="utf-8", errors="ignore")
    last_err = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:

            raw_output = text_agent(prompt, nougat)
            pred_json  = json.loads(raw_output)

            pred_file.write_text(
                json.dumps(pred_json, ensure_ascii=False, indent=2),
                encoding="utf-8"
            )

            gt_cells = parse_gt_json(gt_json_path, normalize=True)
            pred_cells = parse_pred_json(pred_json)

            return {
                "split": split_name, 
                "stem": stem,
                "image_path": str(img_path), 
                "gt_path": str(gt_json_path),
                "num_gt_cells": len(gt_cells), 
                "num_pred_cells": len(pred_cells),
                "status": "ok",
            }

        except Exception as e:
            last_err = e
            print(f"  [attempt {attempt}] Error for {stem}: {repr(last_err)}")
            time.sleep(RETRY_BACKOFF_SEC * attempt)

    err_path = errs_dir / f"{stem}.txt"
    msg = f"ERROR for {stem}\nIMG: {img_path}\nGT:  {gt_json_path}\n\n{repr(last_err)}\n"

    if raw_output:
        if hasattr(raw_output, "model_dump_json"):
            msg += "\n\n=== RAW MODEL OUTPUT ===\n" + raw_output.model_dump_json(indent=2)
        else:
            msg += "\n\n=== RAW MODEL OUTPUT ===\n" + str(raw_output)
    err_path.write_text(msg, encoding="utf-8")

    return {
        "split": split_name, 
        "stem": stem,
        "image_path": str(img_path), 
        "gt_path": str(gt_json_path),
        "num_gt_cells": 0, 
        "num_pred_cells": 0,
        "status": "error",
    }

In [ ]:
def run_split_sequential(split_name: str):
    pairs   = list_pairs_for_split(split_name)
    out_dir = OUT_DIR / split_name
    out_dir.mkdir(parents=True, exist_ok=True)

    preds_dir = out_dir / "predictions"
    errs_dir  = out_dir / "errors"
    preds_dir.mkdir(exist_ok=True)
    errs_dir.mkdir(exist_ok=True)

    results_csv = out_dir / "results.csv"
    done = set()

    if results_csv.exists():
        with open(results_csv, "r", newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                done.add(row["stem"])

    write_header = not results_csv.exists()
    pairs_to_process = [p for p in pairs if p[0].stem not in done]

    if not pairs_to_process:
        print(f"All tables in '{split_name}' already processed.")
        return results_csv

    fieldnames = [
        "split",
        "stem", 
        "image_path", 
        "gt_path",
        "num_gt_cells", 
        "num_pred_cells",
        "status",
    ]

    with open(results_csv, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()

        for img_path, gt_path in tqdm(pairs_to_process, desc=f"{split_name} (Sequential)"):
            row_data = process_pair_sequential(img_path, gt_path, split_name, preds_dir, errs_dir)
            writer.writerow(row_data)
            f.flush()

    return results_csv


split_results = []
for split in SPLITS:
    csv_path = run_split_sequential(split)
    split_results.append(csv_path)

split_results